# Partie II — CNN et vision par ordinateur
### Classification d'images réelles (Fashion-MNIST) et analyse des opérations convolutionnelles

**Étudiant·e :** Malak — EMSI, promotion 2025/2026
**Module :** Deep Learning — Partie II (35 points)

Ce notebook couvre, dans l'ordre du cahier des charges :
1. Pourquoi un MLP est mal adapté aux images / idées fondatrices des CNN
2. Calculs manuels (corrélation croisée 2D, taille de sortie conv/pooling)
3. Implémentation "from scratch" de la corrélation croisée 2D, max-pooling, average-pooling
4. Comparaison avec les couches PyTorch natives
5. CNN inspiré de LeNet
6. Étude expérimentale : padding, stride, pooling, nombre de filtres, conv 1×1
7. Visualisation des cartes de caractéristiques
8. MLP vs CNN sur le même dataset
9. Analyse critique et question de synthèse

> **Dataset :** Fashion-MNIST (60 000 images d'entraînement + 10 000 de test, 28×28
> niveaux de gris, 10 classes de vêtements) — téléchargé automatiquement via
> `torchvision.datasets`, l'un des datasets explicitement proposés par le cahier des charges.

**Exécution :** `Exécution > Tout exécuter` dans Colab (GPU recommandé : Runtime > Change
runtime type > GPU). Le téléchargement du dataset nécessite une connexion internet (Colab
en a une par défaut).

In [ ]:
# 0. Imports et configuration
import time
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


## 1. Pourquoi un MLP est-il peu adapté aux images ?

- **Explosion du nombre de paramètres** : un MLP "aplatit" l'image (28×28 = 784 pixels)
  et connecte *chaque* pixel à *chaque* neurone de la première couche cachée. Pour une
  image 224×224 (cas réaliste), la première couche compterait déjà des dizaines de
  millions de poids — avant même de parler de profondeur.
- **Perte de la structure spatiale** : aplatir l'image détruit la notion de voisinage —
  un pixel et son voisin immédiat sont traités exactement comme deux pixels opposés de
  l'image. Le MLP doit *réapprendre* depuis zéro, à partir des données, que les pixels
  proches sont corrélés — alors que c'est une propriété universelle des images naturelles.
- **Pas d'invariance par translation** : un MLP apprend un motif (ex. "bord vertical") à
  un endroit fixe de l'image ; si ce motif apparaît ailleurs au test, il doit être réappris
  avec des poids différents.

### Idées fondatrices des CNN

1. **Localité (champ réceptif local)** : chaque neurone ne regarde qu'une petite fenêtre
   de l'image (le noyau de convolution), pas l'image entière.
2. **Partage des poids (weight sharing)** : le **même** noyau glisse sur toute l'image →
   un motif appris à un endroit est automatiquement détecté partout ailleurs (invariance
   par translation), et le nombre de paramètres ne dépend plus de la taille de l'image.
3. **Hiérarchie des représentations** : les premières couches détectent des motifs simples
   (bords, contrastes), les couches profondes combinent ces motifs en concepts de plus haut
   niveau (formes, parties d'objet, objets entiers) — une hiérarchie compositionnelle.

## 2. Calculs manuels — corrélation croisée 2D et tailles de sortie

**Corrélation croisée 2D** (ce que PyTorch appelle "convolution" par abus de langage,
sans retournement du noyau) : pour une entrée $X$ de taille $n_h \times n_w$ et un noyau
$K$ de taille $k_h \times k_w$, la sortie en position $(i,j)$ est :

$$Y_{i,j} = \sum_{a=0}^{k_h-1} \sum_{b=0}^{k_w-1} X_{i+a,\, j+b} \cdot K_{a,b}$$

**Taille de sortie** avec padding $p$ et stride $s$, noyau $k$, entrée $n$ :

$$n_{out} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1$$

**Exemple manuel** : $X$ de taille $4\times4$, noyau $3\times3$, $p=0$, $s=1$
→ $n_{out} = \lfloor (4 + 0 - 3)/1 \rfloor + 1 = 2$, donc une sortie $2\times2$.

In [ ]:
# 2.1 Calcul manuel de corrélation croisée 2D sur un petit exemple "à la main"
X_demo = torch.tensor([[1., 2., 3., 0.],
                        [4., 5., 6., 1.],
                        [7., 8., 9., 2.],
                        [0., 1., 2., 3.]])
K_demo = torch.tensor([[1., 0., -1.],
                        [1., 0., -1.],
                        [1., 0., -1.]])  # détecteur de bord vertical simple

# Calcul "à la main" pour la position (0,0) :
manual_00 = (X_demo[0:3, 0:3] * K_demo).sum()
print("Y[0,0] calculé à la main :", manual_00.item())
print("(à comparer avec la fonction cross_correlation_2d ci-dessous)")


In [ ]:
# 2.2 Table des tailles de sortie pour plusieurs combinaisons (n, k, p, s)
def output_size(n, k, p, s):
    return (n + 2 * p - k) // s + 1

configs = [
    (28, 3, 0, 1), (28, 3, 1, 1), (28, 5, 0, 1), (28, 5, 2, 1),
    (28, 3, 0, 2), (28, 3, 1, 2), (28, 2, 0, 2),
]
rows = [{"n_in": n, "kernel": k, "padding": p, "stride": s, "n_out": output_size(n, k, p, s)}
        for (n, k, p, s) in configs]
df_sizes = pd.DataFrame(rows)
df_sizes


## 3. Implémentation "from scratch" : corrélation croisée, max-pooling, average-pooling

On implémente ces trois opérations **sans** utiliser `nn.Conv2d` / `nn.MaxPool2d`, puis on
vérifie qu'elles produisent les mêmes résultats que les couches PyTorch natives
(`F.conv2d`, `F.max_pool2d`, `F.avg_pool2d`) — preuve de correction par comparaison directe.

In [ ]:
def cross_correlation_2d(X, K):
    """Corrélation croisée 2D 'from scratch', sans padding, stride=1.
    X : tenseur (H, W). K : noyau (kh, kw). Retourne Y (H-kh+1, W-kw+1)."""
    kh, kw = K.shape
    h, w = X.shape
    out_h, out_w = h - kh + 1, w - kw + 1
    Y = torch.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            Y[i, j] = (X[i:i + kh, j:j + kw] * K).sum()
    return Y


def max_pool_2d_manual(X, pool_size, stride):
    """Max-pooling 'from scratch'. X : (H, W)."""
    h, w = X.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    Y = torch.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            i0, j0 = i * stride, j * stride
            Y[i, j] = X[i0:i0 + pool_size, j0:j0 + pool_size].max()
    return Y


def avg_pool_2d_manual(X, pool_size, stride):
    """Average-pooling 'from scratch'. X : (H, W)."""
    h, w = X.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    Y = torch.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            i0, j0 = i * stride, j * stride
            Y[i, j] = X[i0:i0 + pool_size, j0:j0 + pool_size].mean()
    return Y


In [ ]:
# 3.1 Vérification : corrélation croisée maison vs F.conv2d (avec noyau retourné pour
#     compenser le fait que conv2d retourne le noyau par convention historique -- en
#     pratique pour le deep learning, PyTorch conv2d == corrélation croisée, pas besoin
#     de retournement)
Y_manual = cross_correlation_2d(X_demo, K_demo)

X_4d = X_demo.unsqueeze(0).unsqueeze(0)   # (1,1,H,W)
K_4d = K_demo.unsqueeze(0).unsqueeze(0)   # (1,1,kh,kw)
Y_torch = F.conv2d(X_4d, K_4d).squeeze()

print("Sortie maison :\n", Y_manual)
print("\nSortie F.conv2d :\n", Y_torch)
print("\nDifférence max :", (Y_manual - Y_torch).abs().max().item())
assert torch.allclose(Y_manual, Y_torch, atol=1e-5)
print("OK -- implémentation maison validée contre PyTorch.")


In [ ]:
# 3.2 Vérification max-pooling et average-pooling maison vs PyTorch
torch.manual_seed(0)
X_pool_demo = torch.rand(8, 8)

Ymax_manual = max_pool_2d_manual(X_pool_demo, pool_size=2, stride=2)
Ymax_torch = F.max_pool2d(X_pool_demo.unsqueeze(0).unsqueeze(0), kernel_size=2, stride=2).squeeze()
print("Max-pool -- écart max :", (Ymax_manual - Ymax_torch).abs().max().item())
assert torch.allclose(Ymax_manual, Ymax_torch, atol=1e-5)

Yavg_manual = avg_pool_2d_manual(X_pool_demo, pool_size=2, stride=2)
Yavg_torch = F.avg_pool2d(X_pool_demo.unsqueeze(0).unsqueeze(0), kernel_size=2, stride=2).squeeze()
print("Avg-pool -- écart max :", (Yavg_manual - Yavg_torch).abs().max().item())
assert torch.allclose(Yavg_manual, Yavg_torch, atol=1e-5)

print("OK -- max-pooling et average-pooling maison validés contre PyTorch.")


## 4. Chargement du dataset réel et préparation des DataLoaders

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                         # [0,255] -> [0,1], shape (1,28,28)
    transforms.Normalize((0.2860,), (0.3530,)),     # moyenne/std officielles Fashion-MNIST
])

train_full = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_set = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

# Split train -> train/val (90/10)
n_val = int(0.1 * len(train_full))
n_train = len(train_full) - n_val
train_set, val_set = torch.utils.data.random_split(
    train_full, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train : {len(train_set)} | Val : {len(val_set)} | Test : {len(test_set)}")


In [ ]:
# 4.1 Visualisation de quelques exemples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_full[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")
plt.suptitle("Exemples — Fashion-MNIST")
plt.tight_layout()
plt.show()


## 5. CNN inspiré de LeNet (architecture paramétrable)

Architecture configurable pour pouvoir mener l'étude expérimentale de la section 6 :
`Conv(1->f1) -> [Conv1x1 optionnel] -> Activation -> Pool -> Conv(f1->f2) -> Activation ->
Pool -> Flatten -> FC -> FC(10)`, avec `padding`, `stride`, `pool_type` et `num_filters`
en paramètres.

In [ ]:
class ConfigurableCNN(nn.Module):
    def __init__(self, num_filters=(6, 16), kernel_size=5, padding=0, stride=1,
                 pool_type="max", use_1x1conv=False, num_classes=10):
        super().__init__()
        f1, f2 = num_filters
        self.conv1 = nn.Conv2d(1, f1, kernel_size=kernel_size, padding=padding, stride=stride)
        self.conv1x1 = nn.Conv2d(f1, f1, kernel_size=1) if use_1x1conv else nn.Identity()
        self.conv2 = nn.Conv2d(f1, f2, kernel_size=kernel_size, padding=padding, stride=stride)

        self.pool = nn.MaxPool2d(2, 2) if pool_type == "max" else nn.AvgPool2d(2, 2)
        self.act = nn.ReLU()

        # On calcule dynamiquement la taille du flatten avec un passage à blanc
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 28, 28)
            out = self._features(dummy)
            flat_dim = out.view(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def _features(self, x):
        x = self.act(self.conv1(x))
        x = self.conv1x1(x)
        x = self.pool(x)
        x = self.act(self.conv2(x))
        x = self.pool(x)
        return x

    def forward(self, x):
        x = self._features(x)
        x = x.flatten(1)
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)
        return x

# Architecture de référence (baseline)
baseline_cnn = ConfigurableCNN(num_filters=(6, 16), kernel_size=5, padding=0, stride=1,
                                pool_type="max", use_1x1conv=False).to(device)
print(baseline_cnn)
print("Nombre de paramètres :", sum(p.numel() for p in baseline_cnn.parameters()))


## 6. Boucle d'entraînement et fonction d'évaluation génériques

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, correct / total


def train_cnn(model, train_loader, val_loader, n_epochs=8, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    start = time.time()
    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            n += xb.size(0)

        train_loss = running_loss / n
        val_loss, val_acc = evaluate(model, val_loader)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch}/{n_epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    elapsed = time.time() - start
    return history, elapsed


In [ ]:
# 6.1 Entraînement du modèle baseline
baseline_history, baseline_time = train_cnn(baseline_cnn, train_loader, val_loader, n_epochs=8)
print(f"Temps d'entraînement baseline : {baseline_time:.1f}s")


## 7. Étude expérimentale comparative

On fait varier, **un facteur à la fois** par rapport à la baseline ci-dessus, pour isoler
l'effet de chaque choix architectural :
- **padding** : 0 (baseline) vs 2 (préserve davantage les bords)
- **stride** : 1 (baseline) vs 2 (sous-échantillonnage agressif dès la conv)
- **pooling** : max (baseline) vs average
- **nombre de filtres** : (6,16) baseline vs (16,32) (plus capacité)
- **conv 1×1** : absente (baseline) vs présente (mélange les canaux sans changer la résolution)

⚠️ Cellule longue à exécuter (5 entraînements supplémentaires) — réduire `n_epochs` si besoin
de résultats plus rapides pour itérer.

In [ ]:
experiment_configs = {
    "baseline (p=0, s=1, max, f=(6,16), no 1x1)": dict(num_filters=(6, 16), kernel_size=5, padding=0, stride=1, pool_type="max", use_1x1conv=False),
    "padding=2": dict(num_filters=(6, 16), kernel_size=5, padding=2, stride=1, pool_type="max", use_1x1conv=False),
    "stride=2": dict(num_filters=(6, 16), kernel_size=5, padding=0, stride=2, pool_type="max", use_1x1conv=False),
    "avg pooling": dict(num_filters=(6, 16), kernel_size=5, padding=0, stride=1, pool_type="avg", use_1x1conv=False),
    "plus de filtres (16,32)": dict(num_filters=(16, 32), kernel_size=5, padding=0, stride=1, pool_type="max", use_1x1conv=False),
    "avec conv 1x1": dict(num_filters=(6, 16), kernel_size=5, padding=0, stride=1, pool_type="max", use_1x1conv=True),
}

results = []
for name, cfg in experiment_configs.items():
    torch.manual_seed(SEED)
    model = ConfigurableCNN(**cfg, num_classes=10).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    history, elapsed = train_cnn(model, train_loader, val_loader, n_epochs=8)
    test_loss, test_acc = evaluate(model, test_loader)
    results.append({
        "config": name, "n_params": n_params, "train_time_s": round(elapsed, 1),
        "val_acc_final": round(history["val_acc"][-1], 4), "test_acc": round(test_acc, 4),
    })
    print(f"--- {name} -> test_acc={test_acc:.4f} ---\n")

df_results = pd.DataFrame(results).sort_values("test_acc", ascending=False)
df_results


In [ ]:
# 7.1 Visualisation comparative
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_results["config"], df_results["test_acc"], color="#2980b9")
ax.set_xlabel("Accuracy (test)")
ax.set_title("Comparaison des variantes architecturales")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 8. Visualisation et interprétation des cartes de caractéristiques

In [ ]:
# On récupère les activations du premier conv du modèle baseline sur une image de test
baseline_cnn.eval()
sample_img, sample_label = test_set[0]
sample_img_b = sample_img.unsqueeze(0).to(device)

with torch.no_grad():
    feature_maps = baseline_cnn.act(baseline_cnn.conv1(sample_img_b)).squeeze(0).cpu()

n_show = min(6, feature_maps.shape[0])
fig, axes = plt.subplots(1, n_show + 1, figsize=(3 * (n_show + 1), 3))
axes[0].imshow(sample_img.squeeze(), cmap="gray")
axes[0].set_title(f"Entrée\n({CLASS_NAMES[sample_label]})")
axes[0].axis("off")
for i in range(n_show):
    axes[i + 1].imshow(feature_maps[i], cmap="viridis")
    axes[i + 1].set_title(f"Filtre {i}")
    axes[i + 1].axis("off")
plt.suptitle("Cartes de caractéristiques -- premier bloc convolutionnel")
plt.tight_layout()
plt.show()


**Interprétation attendue** *(à ajuster avec ce que tu observes réellement)* : les premiers
filtres réagissent typiquement à des contrastes locaux et à des orientations de bords (haut
clair / bas sombre, gauche/droite). Sur un réseau peu profond comme celui-ci, on ne s'attend
pas encore à des détecteurs de "parties d'objets" — cette hiérarchie se construit avec la
profondeur, ce qui motive l'usage d'architectures plus profondes en pratique.

## 9. Comparaison MLP simple vs CNN sur le même dataset

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=28 * 28, hidden=256, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(SEED)
mlp_baseline = SimpleMLP().to(device)
print("Paramètres MLP :", sum(p.numel() for p in mlp_baseline.parameters()))
print("Paramètres CNN baseline :", sum(p.numel() for p in baseline_cnn.parameters()))

mlp_history, mlp_time = train_cnn(mlp_baseline, train_loader, val_loader, n_epochs=8)
mlp_test_loss, mlp_test_acc = evaluate(mlp_baseline, test_loader)
cnn_test_loss, cnn_test_acc = evaluate(baseline_cnn, test_loader)

comparison = pd.DataFrame([
    {"modèle": "MLP simple", "n_params": sum(p.numel() for p in mlp_baseline.parameters()),
     "train_time_s": round(mlp_time, 1), "test_acc": round(mlp_test_acc, 4)},
    {"modèle": "CNN baseline (LeNet-like)", "n_params": sum(p.numel() for p in baseline_cnn.parameters()),
     "train_time_s": round(baseline_time, 1), "test_acc": round(cnn_test_acc, 4)},
])
comparison


## 10. Analyse critique *(template — à compléter avec tes chiffres réels)*

- **Effet du padding/stride/pooling** : [à compléter à partir du tableau de la section 7 —
  ex. "le padding=2 améliore/dégrade l'accuracy de X points car ___"].
- **Effet du nombre de filtres** : plus de filtres = plus de capacité de représentation,
  au prix d'un temps d'entraînement et d'un risque de surapprentissage plus élevés — à
  quantifier avec `n_params` et `train_time_s` du tableau.
- **Effet de la conv 1×1** : elle mélange les canaux sans changer la résolution spatiale ;
  son intérêt principal est la réduction/augmentation du nombre de canaux à coût de calcul
  minimal (utile dans des architectures profondes type Inception/ResNet, moins déterminant
  sur un réseau aussi peu profond que celui-ci).
- **MLP vs CNN** : écart d'accuracy de **___** points en faveur du CNN (section 9), pour un
  nombre de paramètres [comparable / inférieur / supérieur] — preuve empirique que l'a priori
  structurel des CNN (localité + partage de poids) est plus efficace que la flexibilité brute
  du MLP, *spécifiquement parce que* les images ont une structure spatiale exploitable.
- **Limites** : 8 epochs seulement (par souci de temps d'exécution) — les classements
  pourraient se resserrer avec un entraînement plus long ou de la data augmentation.

## 11. Question de synthèse — Partie II *(template à compléter)*

> *Pourquoi un CNN est-il plus pertinent qu'un MLP pour une tâche de classification d'images
> sur un dataset réel, et comment les choix de padding, stride, pooling et profondeur
> influencent-ils réellement les performances du modèle ?*

**Plan de réponse à rédiger :**
1. **Calculs dimensionnels** — relier la formule de taille de sortie (section 2) aux choix
   concrets testés en section 7 (ex. stride=2 réduit la résolution deux fois plus vite que
   stride=1, ce qui réduit le nombre de paramètres des couches FC en aval mais perd de
   l'information spatiale fine).
2. **Théorie des CNN** — localité + partage de poids (section 1) expliquent la supériorité
   structurelle sur des données à structure spatiale.
3. **Résultats expérimentaux** — citer les chiffres précis du tableau comparatif (section 7)
   et de la comparaison MLP/CNN (section 9).
4. **Interprétation des représentations internes** — relier aux cartes de caractéristiques
   visualisées (section 8) : les filtres bas niveau détectent des primitives visuelles
   générales, justifiant l'argument de hiérarchie compositionnelle.